# RAG and NLP systems: raw data to program validation

This notebook creates a deliberately disorganized sentiment dataset, recovers the canonical records, trains and evaluates the real sentiment pipeline, builds a persisted retrieval index, asks a grounded question, and finishes by running the repository tests.

The defects are deterministic: duplicates, blank rows, inconsistent label case, leading/trailing whitespace, repeated internal whitespace, and invalid labels. The clean reference is used only to generate known truth and verify that cleaning did not silently change meaning.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Run this notebook from the repository root."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "notebooks" / "data"
ARTIFACT_DIR = ROOT / "notebooks" / "artifacts"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository: {ROOT.name}")
print(f"Python: {sys.version.split()[0]}")

In [ ]:
import csv
import random
import re

canonical_path = ROOT / "data" / "sentiment.csv"
with canonical_path.open(newline="", encoding="utf-8") as handle:
    canonical = list(csv.DictReader(handle))

messy = []
for index, row in enumerate(canonical):
    text = row["text"]
    if index % 5 == 0:
        text = "   " + text.replace(" ", "   ") + "   "
    label = row["label"]
    if index % 4 == 0:
        label = f" {label.upper()} "
    messy.append({"text": text, "label": label, "import_note": "legacy export"})

messy.extend(messy[:6])
messy.extend(
    [
        {"text": "", "label": "positive", "import_note": "blank text"},
        {"text": "ambiguous example", "label": "mixed", "import_note": "invalid label"},
        {"text": "   ", "label": "negative", "import_note": "whitespace text"},
    ]
)
random.Random(42).shuffle(messy)

messy_path = DATA_DIR / "messy_sentiment.csv"
with messy_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["text", "label", "import_note"])
    writer.writeheader()
    writer.writerows(messy)

print(f"Canonical rows: {len(canonical)}")
print(f"Messy rows: {len(messy)}")
print(f"Messy file: {messy_path.relative_to(ROOT)}")
messy[:4]

In [ ]:
cleaned_by_text = {}
rejected = []
for row in messy:
    text = re.sub(r"\s+", " ", str(row.get("text", ""))).strip()
    label = str(row.get("label", "")).strip().lower()
    if not text or label not in {"positive", "negative"}:
        rejected.append(row)
        continue
    cleaned_by_text.setdefault(text.casefold(), {"text": text, "label": label})

recovered_keys = set(cleaned_by_text)
expected_keys = {row["text"].casefold() for row in canonical}
assert recovered_keys == expected_keys
cleaned = [cleaned_by_text[row["text"].casefold()] for row in canonical]
clean_path = DATA_DIR / "cleaned_sentiment.csv"
with clean_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["text", "label"])
    writer.writeheader()
    writer.writerows(cleaned)

assert cleaned == canonical
assert len(rejected) == 3
print(f"Recovered all {len(cleaned)} canonical rows; rejected {len(rejected)} invalid rows.")

In [ ]:
from rag_nlp_systems.sentiment import SentimentPipeline, load_labeled_csv

texts, labels = load_labeled_csv(clean_path)
sentiment = SentimentPipeline()
evaluation = sentiment.evaluate(texts, labels)
model_path = sentiment.fit(texts, labels).save(ARTIFACT_DIR / "sentiment.joblib")
prediction = sentiment.predict(["The technical review was accurate and reproducible"])[0]

assert evaluation.accuracy >= 0.70
assert model_path.is_file()
assert prediction["label"] in {"positive", "negative"}
print({"accuracy": evaluation.accuracy, "prediction": prediction})

In [ ]:
from rag_nlp_systems.rag import KnowledgeBase, RAGAgent

knowledge = KnowledgeBase.from_directory(ROOT / "data" / "knowledge")
index_path = knowledge.save(ARTIFACT_DIR / "knowledge.joblib")
reloaded = KnowledgeBase.load(index_path)
answer = RAGAgent(reloaded).answer("How does batching affect inference capacity?")

assert answer.citations
assert any(result.score > 0 for result in answer.results)
print({"answer": answer.answer, "citations": answer.citations})

In [ ]:
completed = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(completed.stdout)
print(completed.stderr)
assert completed.returncode == 0, "Repository tests failed"
print("VALIDATED: the repository test suite passes in this UV environment.")